# A.4 答案能回到來源嗎？


答案說「紅街9號〔d2〕」，d2卻寫「青街8號」。編號真的存在，內容仍沒得到支持。引用需要兩層核對：先找得回這個編號，再看原文是否支持回答說的每件事。

引用有效性只查來源存在；支持性則查能否由原文得出主張。文件有地址，不代表同時支持店主名字或價格。下面只限制一個肯定地址欄位，方便把兩層分開看。


In [1]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [2]:
source = {"id": "d2", "text": "書店地址是青街8號"}
answers = [
    {"address": "青街8號", "citation": "d2"},
    {"address": "紅街9號", "citation": "d2"},
]
for answer in answers:
    valid = answer["citation"] == source["id"]
    supports = valid and answer["address"] in source["text"]
    print(answer["address"], "引用存在", valid, "地址在原文", supports)

青街8號 引用存在 True 地址在原文 True
紅街9號 引用存在 True 地址在原文 False


青街8號得True、True；紅街9號得True、False。`and`先確定編號相符，`in`再查該地址字串。把第一份引用改成d9，兩項都False，碰巧答對地址也沒有完成回查。

字串出現只是這個窄例的替代判準。原文若寫「不是青街8號」，同字串仍不支持肯定回答；同義說法也可能沒有逐字吻合。正式資料保留店名、日期、完整原文和逐項主張，依否定、條件與版本核對。

舊模型也曾對K003引用存在的D9，卻取了另一家K013的地址C9，輸出 `C9[D9]`。錯誤因此不是編號找不到，而是答案與該來源不相符。來源有格式與來源能支持答案，不能混在一起。

<details>
<summary>補充：舊公告生成的支持檢查範圍</summary>

正式生成也出現這個錯誤：問`K003`，提示裡`D9`寫`K003 address=D2`，另一份`D0`寫`K013 address=C9`；模型卻答`C9[D9]`。`D9`確實存在，但其中沒有它聲稱的地址。加入正確公告與幹擾公告後，12/12回答都列了存在的來源，只有2/12的地址獲所引公告支持，也只有2/12符合原題。引用格式成功因此遠不足以判成功。

這份支持檢查只核對公告中肯定的`address=`欄位，不處理否定、日期可靠性或自然語言蘊涵。另外，只有別家公告時五份回答也能獲那份公告支持，卻沒有一份回答原店的新地址；「有來源支持」還要和「回答了當前問題」分開看。

</details>
